# मानव-इन-द-लूप: पूर्व-कर्म द्वार, जोखिम स्तरीकरण, और ऑडिट लॉगिंग

इस पाठ के README में मानव-इन-द-लूप को एक छोटे से स्निपेट के साथ प्रस्तुत किया गया है जो उपयोगकर्ता से `APPROVE` या `REJECT` पूछता है जब एजेंट पहले ही एक प्रतिक्रिया दे चुका होता है। यह पैटर्न एक अच्छा प्रारंभिक बिंदु है, लेकिन उत्पादन HITL कार्यान्वयन में आमतौर पर तीन अतिरिक्त चीजों की आवश्यकता होती है:

1. एक **पूर्व-कर्म द्वार** जो एजेंट के जोखिमपूर्ण कदम उठाने से **पहले** चलता है, ताकि लागत, अपरिवर्तनीयता, और विलंब नियंत्रण में रह सकें।
2. **जोखिम स्तरीकरण**, ताकि निम्न-जोखिम वाले क्रियाएं स्वतः चल सकें, मध्यम-जोखिम वाली क्रियाएं बैच-स्वीकृत हों, और केवल उच्च-जोखिम वाली क्रियाएं मानव पर निर्भर हों।
3. एक **ऑडिट लॉग और संशोधन चक्र**, ताकि हर द्वार निर्णय को JSONL के रूप में रिकॉर्ड किया जाए, और अस्वीकृति एजेंट को केवल `Revising...` प्रदर्शित करने के बजाय संरचित कारण के साथ पुनः प्रॉम्प्ट करे।

यह नोटबुक इन्हें `06-system-message-framework.ipynb` जैसी ही मूलभूत बातों के ऊपर बनाता है। यह `DEMO_MODE = True` में एंड-टू-एंड चलता है (कोई इंटरैक्टिव इनपुट आवश्यक नहीं) या वास्तविक `input()` प्रॉम्प्ट के साथ जब `DEMO_MODE = False` हो। ध्यान दें: DEMO_MODE में तीसरे लक्ष्य की पुनः कोशिश स्क्रिप्ट की गई है ताकि लूप की प्रक्रियाएँ एंड-टू-एंड दृश्य हो सकें। वास्तविक संशोधन-संचालित पुन: वर्गीकरण के लिए `DEMO_MODE = False` और एक परिचालक की आवश्यकता होती है।

**आउट ऑफ स्कोप (अन्य पाठों में संभाला गया):** प्रमाणीकरण और पहुँच नियंत्रण (पाठ 06 README खतरा 2), टूल-कॉल मिडलवेयर (पाठ 14 MAF गहराई से), मल्टी-एजेंट बहस पैटर्न।


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## पैटर्न 1: पूर्व-क्रिया गेट

README का HITL स्निपेट पहले एजेंट को कॉल करता है, फिर उपयोगकर्ता से आउटपुट को अनुमोदित करने के लिए कहता है। यह एक **पोस्ट-क्रिया** प्रवाह है। एजेंट पहले ही निष्पादित हो चुका है, इसलिए LLM कॉल का खर्च पहले ही चुका दिया गया है, और कोई भी साइड इफेक्ट (भेजा हुआ ईमेल, लिखा गया डेटाबेस रो, पोस्ट किया गया टिप्पणी) पहले ही हो चुका है।

एक **पूर्व-क्रिया** प्रवाह जोखिम भरे कदम को चलाने से पहले गेट डालता है। एजेंट क्रिया का प्रस्ताव करता है, गेट निर्णय लेता है कि इसे निष्पादित करना है या नहीं, और केवल अनुमोदन पर ही साइड इफेक्ट होता है।

| पहलू | पोस्ट-क्रिया अनुमोदन (README स्निपेट) | पूर्व-क्रिया गेट (इस नोटबुक) |
|---|---|---|
| अनुमोदन कब चलता है? | आउटपुट के उत्पादन के बाद | किसी भी साइड-इफेक्ट के निष्पादन से पहले |
| अस्वीकृति पर LLM लागत | पहले ही चुका दी गई | केवल प्रस्ताव के लिए भुगतान, क्रिया के लिए नहीं |
| अपरिवर्तनीय साइड प्रभाव | संभव (क्रिया पहले ही हो चुकी है) | रोका गया |
| ऑडिट स्पष्टता | अनुमोदन एक प्रिंट स्टेटमेंट है | अनुमोदन एक JSON रिकॉर्ड है जिसमें टाइमस्टैम्प, क्रिया, कारण होता है |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## पैटर्न 2: जोखिम स्तर निर्धारण

हर कार्रवाई को मानव की मंजूरी की जरूरत नहीं होती। एक सार्वजनिक API के खिलाफ केवल पढ़ने वाला लुकअप की स्थिति ग्राहक ईमेल भेजने जैसी नहीं होती। दोनों को समान रूप से ट्रीट करना ऑपरेटर का ध्यान बर्बाद करता है और एजेंट को धीमा कर देता है।

एक सरल 3-स्तरीय मॉडल:

| स्तर | उदाहरण | मंजूरी प्रवाह |
|---|---|---|
| `low` (केवल पढ़ने वाला) | ज्ञान आधार खोजें, उड़ान विकल्प देखें, सार्वजनिक वेबपेज प्राप्त करें | स्वचालित निष्पादन, ऑडिट के लिए लॉग किया गया |
| `medium` (सस्ता परिवर्तन) | परिणाम कैश करें, काउंटर बढ़ाएं, अनुस्मारक सेट करें | स्वचालित निष्पादन, लेकिन दैनिक समीक्षा में पैक किया गया |
| `high` (बाहरी-उन्मुख या अपरिवर्तनीय) | ईमेल भेजें, कार्ड चार्ज करें, सार्वजनिक चैनल पर पोस्ट करें | मानव अनुमोदन पर अवरुद्ध करें |

यह एक स्तर निर्धारण है। प्रोडक्शन सिस्टम अक्सर अधिक सूक्ष्म स्तरों का उपयोग करते हैं (जैसे, AWS IAM अनुमति स्तर, भूमिका-आधारित एक्सेस स्तर)। नीचे दिया गया 3-स्तरीय संस्करण एक एजेंट के लिए सबसे छोटा उपयोगी संस्करण है जो केवल पढ़ने वाले और साइड-इफेक्टिंग क्रियाओं को मिश्रित करता है।

नीचे दिया गया वर्गीकरणकर्ता कीवर्ड युक्तियों का उपयोग करता है ताकि डेमो निर्धारक और सस्ता रहे। एक प्रोडक्शन सिस्टम में आप इसे सीखना वाला वर्गीकरणकर्ता या नीति इंजन से बदलेंगे।


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## पैटर्न 3: ऑडिट लॉग और संशोधन लूप

एक `print("Response approved.")` ऑडिट लॉग नहीं है। भरोसे के लिए, हर गेट निर्णय को एक संरचित इवेंट के रूप में रिकॉर्ड किया जाना चाहिए जिसे आप बाद में क्वेरी कर सकें, फिर से चला सकें, या किसी घटना समीक्षा से जोड़ सकें।

दो भाग:

1. **केवल जोड़ने योग्य JSONL।** प्रत्येक निर्णय के लिए एक पंक्ति, जिसमें टाइमस्टैम्प, क्रिया, स्तरीय, निर्णय, कारण होता है। grep करना आसान, बाद में एक वास्तविक लॉग स्टोर में भेजना आसान।
2. **पतन पर संशोधन लूप।** जब गेट `deny` लौटाता है, तो एजेंट अपने आप को अस्वीकृति कारण के साथ पुनः प्रॉम्प्ट करता है, ताकि अगली प्रस्तावना समस्या से बच सके।


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## अतिरिक्त संसाधन

कई अन्य सार्वजनिक परियोजनाएं इन HITL पैटर्न के विभिन्न संस्करणों को लागू करती हैं। अपने स्टैक के लिए उपयुक्त विधि खोजने के लिए दृष्टिकोणों की तुलना करें:

- **LangChain** ह्यूमन-इन-द-लूप टूल रैपर ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ऐसे ड्रॉप-इन टूल रैपर जो मानवीय इनपुट के लिए निष्पादन को रोकते हैं।
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ने इसे पुनः संरचित किया): मल्टी-एजेंट वार्तालाप में मानव का प्रतिनिधित्व करने के लिए एक एजेंट भूमिका का उपयोग करता है।
- **Microsoft Agent Framework (MAF)** फंक्शन-इनवोकेशन मिडलवेयर ([docs](https://learn.microsoft.com/agent-framework/)): ऐसा मिडलवेयर जो हर टूल/फ़ंक्शन कॉल के चारों ओर चलता है, जो गेटिंग लॉजिक और अनुमोदन प्रवाह के लिए उपयुक्त है।

प्रत्येक परियोजना इन तीन उप-पैटर्न को अलग-अलग तरीके से संभालती है: LangChain इन्हें टूल के रूप में रैप करता है, AutoGen एजेंट भूमिका का उपयोग करता है, और Microsoft Agent Framework फंक्शन-इनवोकेशन मिडलवेयर का उपयोग करता है। अपने एजेंट के लिए डिज़ाइन चुनने से पहले एक या दो कार्यान्वयन को शुरू से अंत तक पढ़ें।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
